# 🍔 FOODPULSE
## Food Delivery Intelligence using Apache Spark

**Big Data Analytics Lab — Academic Year 2026–27**

**Students:** Nikeeta Gurve (Roll No. 9) · Sara Joshi (Roll No. 15)  
**Guide:** Sonam Chopade  
**Course:** Big Data Analytics Lab · **Course Code:** 24CS02PR0503

### Dataset provenance
This notebook uses the **public Zomato Food Delivery Operations dataset** published in the repository below. The repository documents **45,584 records and 20 raw attributes** covering delivery partners, locations, order time, weather, traffic, vehicle condition, order type, multiple deliveries, festival indicator, city and delivery time.

**Primary source repository:** https://github.com/Varunnagar/Zomato-Food-Delivery-Analysis

> **Academic note:** This is a publicly distributed Zomato food-delivery dataset, not an official internal Zomato database. We do not invent revenue, cancellation, restaurant-name, or customer fields that are absent from the source.

### This notebook
- Automatically downloads the source CSV
- Verifies the expected dataset size and schema
- Cleans and validates data with Apache Spark
- Demonstrates RDD, DataFrame and Spark SQL
- Calculates delivery KPIs and operational insights
- Calculates geographic distance from coordinates
- Creates charts
- Exports tables for the FoodPulse Streamlit website


In [ ]:
!pip -q install pyspark==4.0.4 pandas matplotlib seaborn

In [ ]:
import os, re, urllib.request
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("FoodPulse-Food-Delivery-Analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


## 1. Automatic dataset download + verification

The notebook downloads the exact public CSV from the repository above.

**Hard verification:** 45,584 rows × 20 raw columns.  
If those checks fail, the notebook stops instead of silently substituting another dataset.


In [ ]:
DATA_DIR = Path("/content/foodpulse_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/Varunnagar/Zomato-Food-Delivery-Analysis/main/Zomato%20Dataset.csv"
DATA_PATH = DATA_DIR / "Zomato_Dataset.csv"

try:
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Downloaded:", DATA_PATH)
except Exception as e:
    print("Automatic download failed:", e)
    print("Fallback: upload the exact 'Zomato Dataset.csv' from the public source repository.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No dataset uploaded.")
    first_name = next(iter(uploaded))
    DATA_PATH = Path("/content") / first_name

print("Dataset path:", DATA_PATH)


In [ ]:
EXPECTED_ROWS = 45584
EXPECTED_COLS = 20

raw_pd = pd.read_csv(DATA_PATH)
print("Rows:", len(raw_pd))
print("Columns:", len(raw_pd.columns))
print("\nColumns:")
for i, c in enumerate(raw_pd.columns, 1):
    print(f"{i:02d}. {c}")

if len(raw_pd) != EXPECTED_ROWS or len(raw_pd.columns) != EXPECTED_COLS:
    raise ValueError(
        f"DATASET VERIFICATION FAILED: expected {EXPECTED_ROWS} x {EXPECTED_COLS}, "
        f"received {len(raw_pd)} x {len(raw_pd.columns)}."
    )

print("\n✅ Verified: 45,584 rows × 20 raw columns.")


In [ ]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_PATH))
)

print("Spark rows:", df_raw.count())
print("Spark columns:", len(df_raw.columns))
df_raw.printSchema()
display(df_raw.limit(10).toPandas())


## 2. Data cleaning and standardization

The raw dataset is preserved as `df_raw`. A cleaned Spark DataFrame is created for analysis.

Missing values are measured rather than fabricated.


In [ ]:
def clean_col(c):
    c = c.strip()
    c = re.sub(r"[^A-Za-z0-9]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

df = df_raw
for old in df.columns:
    df = df.withColumnRenamed(old, clean_col(old))

numeric_cols = [
    "Delivery_person_Age", "Delivery_person_Ratings",
    "Restaurant_latitude", "Restaurant_longitude",
    "Delivery_location_latitude", "Delivery_location_longitude",
    "Vehicle_condition", "multiple_deliveries",
    "Time_taken_min"
]

for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

df = (
    df.withColumn("Order_Date_parsed", F.to_date("Order_Date", "dd-MM-yyyy"))
      .withColumn("Order_Month", F.month("Order_Date_parsed"))
      .withColumn("Order_DayOfWeek", F.date_format("Order_Date_parsed", "EEEE"))
      .withColumn("Order_WeekOfYear", F.weekofyear("Order_Date_parsed"))
)

for c in ["Weather_conditions", "Road_traffic_density", "Type_of_order",
          "Type_of_vehicle", "Festival", "City"]:
    if c in df.columns:
        df = df.withColumn(c, F.trim(F.col(c)))

df.printSchema()


In [ ]:
null_profile = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
]).toPandas().T.reset_index()
null_profile.columns = ["column", "null_count"]
null_profile["null_pct"] = (null_profile["null_count"] / df.count() * 100).round(2)
display(null_profile.sort_values("null_count", ascending=False))


In [ ]:
duplicate_count = df.count() - df.dropDuplicates(["ID"]).count()
invalid_time_count = df.filter((F.col("Time_taken_min") < 1) | (F.col("Time_taken_min") > 180)).count()
invalid_rating_count = df.filter(
    (F.col("Delivery_person_Ratings") < 1) | (F.col("Delivery_person_Ratings") > 5)
).count()

print("Duplicate ID rows:", duplicate_count)
print("Suspicious delivery-time rows:", invalid_time_count)
print("Out-of-range rating rows:", invalid_rating_count)


## 3. Feature engineering

### Distance
We calculate straight-line geographic distance with the Haversine formula from the supplied restaurant/customer coordinates.

This is **not road distance**.

### Performance bands
Project-defined descriptive bands:
- Fast: <25 min
- Standard: 25–35 min
- Delayed: >35 min

These are not official Zomato SLAs.


In [ ]:
R = 6371.0

lat1 = F.radians(F.col("Restaurant_latitude"))
lon1 = F.radians(F.col("Restaurant_longitude"))
lat2 = F.radians(F.col("Delivery_location_latitude"))
lon2 = F.radians(F.col("Delivery_location_longitude"))

a = (
    F.pow(F.sin((lat2 - lat1) / 2), 2)
    + F.cos(lat1) * F.cos(lat2) * F.pow(F.sin((lon2 - lon1) / 2), 2)
)

df = df.withColumn("Distance_km", 2 * F.lit(R) * F.asin(F.sqrt(a)))

df = df.withColumn(
    "Delivery_Performance",
    F.when(F.col("Time_taken_min") < 25, "Fast")
     .when(F.col("Time_taken_min") <= 35, "Standard")
     .otherwise("Delayed")
)

df = df.withColumn(
    "Delivery_Speed_kmph",
    F.when(
        F.col("Time_taken_min") > 0,
        F.col("Distance_km") / (F.col("Time_taken_min") / 60)
    )
)

display(df.select(
    "ID", "City", "Distance_km", "Time_taken_min",
    "Delivery_Speed_kmph", "Delivery_Performance"
).limit(10).toPandas())


## 4. Core FoodPulse KPIs


In [ ]:
kpis = df.select(
    F.count("*").alias("Total_Orders"),
    F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Time_Min"),
    F.round(F.expr("percentile_approx(Time_taken_min, 0.5)"), 2).alias("Median_Delivery_Time_Min"),
    F.round(F.avg("Delivery_person_Ratings"), 2).alias("Avg_Delivery_Partner_Rating"),
    F.round(F.avg("Distance_km"), 2).alias("Avg_Distance_Km"),
    F.round(F.avg("Delivery_Speed_kmph"), 2).alias("Avg_Estimated_Speed_Kmph")
)
display(kpis.toPandas())


In [ ]:
city_perf = (
    df.groupBy("City")
      .agg(
          F.count("*").alias("Orders"),
          F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Min"),
          F.round(F.expr("percentile_approx(Time_taken_min, 0.5)"), 2).alias("Median_Delivery_Min"),
          F.round(F.avg("Delivery_person_Ratings"), 2).alias("Avg_Rating"),
          F.round(F.avg("Distance_km"), 2).alias("Avg_Distance_Km")
      )
      .orderBy(F.desc("Orders"))
)

traffic_perf = (
    df.groupBy("Road_traffic_density")
      .agg(
          F.count("*").alias("Orders"),
          F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Min"),
          F.round(F.avg("Distance_km"), 2).alias("Avg_Distance_Km")
      )
      .orderBy(F.desc("Orders"))
)

weather_perf = (
    df.groupBy("Weather_conditions")
      .agg(
          F.count("*").alias("Orders"),
          F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Min"),
          F.round(F.avg("Delivery_person_Ratings"), 2).alias("Avg_Rating")
      )
      .orderBy(F.desc("Orders"))
)

vehicle_perf = (
    df.groupBy("Type_of_vehicle")
      .agg(
          F.count("*").alias("Orders"),
          F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Min"),
          F.round(F.avg("Delivery_Speed_kmph"), 2).alias("Avg_Estimated_Speed_Kmph")
      )
      .orderBy(F.desc("Orders"))
)

order_type_perf = (
    df.groupBy("Type_of_order")
      .agg(
          F.count("*").alias("Orders"),
          F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Min")
      )
      .orderBy(F.desc("Orders"))
)

multi_perf = (
    df.groupBy("multiple_deliveries")
      .agg(
          F.count("*").alias("Orders"),
          F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Min"),
          F.round(F.avg("Distance_km"), 2).alias("Avg_Distance_Km")
      )
      .orderBy("multiple_deliveries")
)

festival_perf = (
    df.groupBy("Festival")
      .agg(
          F.count("*").alias("Orders"),
          F.round(F.avg("Time_taken_min"), 2).alias("Avg_Delivery_Min"),
          F.round(F.avg("Delivery_person_Ratings"), 2).alias("Avg_Rating")
      )
      .orderBy(F.desc("Orders"))
)

display(city_perf.toPandas())
display(traffic_perf.toPandas())
display(weather_perf.toPandas())
display(vehicle_perf.toPandas())
display(order_type_perf.toPandas())
display(multi_perf.toPandas())
display(festival_perf.toPandas())


## 5. RDD analytics

This section explicitly demonstrates an RDD transformation and `reduceByKey`, useful for the Big Data Analytics Lab viva.


In [ ]:
city_rdd = (
    df.select("City", "Time_taken_min")
      .dropna()
      .rdd
      .map(lambda r: (r["City"], (1, float(r["Time_taken_min"]))))
      .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
)

rdd_city_summary = [
    (city, values[0], round(values[1] / values[0], 2))
    for city, values in city_rdd.collect()
]

rdd_city_df = pd.DataFrame(
    rdd_city_summary,
    columns=["City", "Orders", "Avg_Delivery_Min"]
).sort_values("Orders", ascending=False)

display(rdd_city_df)


## 6. Spark SQL analytics


In [ ]:
df.createOrReplaceTempView("food_delivery")

sql_city = spark.sql(
    '''
    SELECT City,
           COUNT(*) AS orders,
           ROUND(AVG(Time_taken_min), 2) AS avg_delivery_min,
           ROUND(AVG(Delivery_person_Ratings), 2) AS avg_rating,
           ROUND(AVG(Distance_km), 2) AS avg_distance_km
    FROM food_delivery
    GROUP BY City
    ORDER BY orders DESC
    '''
)

display(sql_city.toPandas())


In [ ]:
df_sql = df.withColumn(
    "Order_Hour",
    F.hour(F.to_timestamp("Time_Orderd", "HH:mm"))
)
df_sql.createOrReplaceTempView("food_delivery_time")

peak_hours = spark.sql(
    '''
    SELECT Order_Hour,
           COUNT(*) AS orders,
           ROUND(AVG(Time_taken_min), 2) AS avg_delivery_min
    FROM food_delivery_time
    WHERE Order_Hour IS NOT NULL
    GROUP BY Order_Hour
    ORDER BY Order_Hour
    '''
)

display(peak_hours.toPandas())


## 7. Visual analytics


In [ ]:
sns.set_theme(style="whitegrid")

city_pd = city_perf.toPandas()
traffic_pd = traffic_perf.toPandas()
weather_pd = weather_perf.toPandas()
vehicle_pd = vehicle_perf.toPandas()
multi_pd = multi_perf.toPandas()
peak_pd = peak_hours.toPandas()


In [ ]:
plt.figure(figsize=(10,5))
tmp = city_pd.sort_values("Avg_Delivery_Min", ascending=False)
sns.barplot(data=tmp, x="Avg_Delivery_Min", y="City")
plt.title("FoodPulse — Average Delivery Time by City")
plt.xlabel("Average delivery time (minutes)")
plt.ylabel("City")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=traffic_pd, x="Road_traffic_density", y="Avg_Delivery_Min")
plt.title("FoodPulse — Traffic Density vs Delivery Time")
plt.xlabel("Traffic density")
plt.ylabel("Average delivery time (minutes)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=weather_pd, x="Weather_conditions", y="Avg_Delivery_Min")
plt.title("FoodPulse — Weather vs Delivery Time")
plt.xlabel("Weather")
plt.ylabel("Average delivery time (minutes)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=vehicle_pd, x="Type_of_vehicle", y="Avg_Delivery_Min")
plt.title("FoodPulse — Vehicle Type vs Delivery Time")
plt.xlabel("Vehicle type")
plt.ylabel("Average delivery time (minutes)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.lineplot(data=multi_pd, x="multiple_deliveries", y="Avg_Delivery_Min", marker="o")
plt.title("FoodPulse — Multiple Deliveries vs Delivery Time")
plt.xlabel("Multiple-delivery count")
plt.ylabel("Average delivery time (minutes)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=peak_pd, x="Order_Hour", y="orders", marker="o")
plt.title("FoodPulse — Orders by Hour")
plt.xlabel("Order hour")
plt.ylabel("Number of orders")
plt.xticks(range(0,24))
plt.tight_layout()
plt.show()


In [ ]:
sample_pd = (
    df.select("Distance_km", "Time_taken_min", "Delivery_person_Ratings",
              "Road_traffic_density", "Weather_conditions")
      .dropna()
      .sample(withReplacement=False, fraction=min(1.0, 5000 / max(df.count(), 1)), seed=42)
      .toPandas()
)

plt.figure(figsize=(9,6))
sns.scatterplot(data=sample_pd, x="Distance_km", y="Time_taken_min", alpha=0.45)
plt.title("FoodPulse — Estimated Distance vs Delivery Time")
plt.xlabel("Estimated straight-line distance (km)")
plt.ylabel("Delivery time (minutes)")
plt.tight_layout()
plt.show()


In [ ]:
corr_cols = [
    "Delivery_person_Age", "Delivery_person_Ratings",
    "Vehicle_condition", "multiple_deliveries",
    "Distance_km", "Delivery_Speed_kmph", "Time_taken_min"
]

corr_pd = df.select(corr_cols).toPandas().corr(numeric_only=True)

plt.figure(figsize=(9,7))
sns.heatmap(corr_pd, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0)
plt.title("FoodPulse — Operational Correlation Matrix")
plt.tight_layout()
plt.show()


## 8. Evidence-based business insights


In [ ]:
k = kpis.toPandas().iloc[0]
slowest_city = city_pd.loc[city_pd["Avg_Delivery_Min"].idxmax()]
fastest_city = city_pd.loc[city_pd["Avg_Delivery_Min"].idxmin()]
slowest_traffic = traffic_pd.loc[traffic_pd["Avg_Delivery_Min"].idxmax()]
slowest_weather = weather_pd.loc[weather_pd["Avg_Delivery_Min"].idxmax()]

print("FOODPULSE — DATA-DRIVEN INSIGHTS")
print("=" * 55)
print(f"• Total analyzed orders: {int(k['Total_Orders']):,}")
print(f"• Average delivery time: {k['Avg_Delivery_Time_Min']:.2f} minutes")
print(f"• Median delivery time: {k['Median_Delivery_Time_Min']:.2f} minutes")
print(f"• Average estimated straight-line distance: {k['Avg_Distance_Km']:.2f} km")
print(f"• Slowest city by average delivery time: {slowest_city['City']} ({slowest_city['Avg_Delivery_Min']:.2f} min)")
print(f"• Fastest city by average delivery time: {fastest_city['City']} ({fastest_city['Avg_Delivery_Min']:.2f} min)")
print(f"• Slowest traffic category: {slowest_traffic['Road_traffic_density']} ({slowest_traffic['Avg_Delivery_Min']:.2f} min)")
print(f"• Highest average delivery time among weather categories: {slowest_weather['Weather_conditions']} ({slowest_weather['Avg_Delivery_Min']:.2f} min)")
print("\nThese are descriptive associations, not proof of causation.")


## 9. Export files for the FoodPulse Streamlit website


In [ ]:
OUT = Path("/content/foodpulse_outputs")
OUT.mkdir(parents=True, exist_ok=True)

exports = {
    "kpis.csv": kpis.toPandas(),
    "city_performance.csv": city_perf.toPandas(),
    "traffic_performance.csv": traffic_perf.toPandas(),
    "weather_performance.csv": weather_perf.toPandas(),
    "vehicle_performance.csv": vehicle_perf.toPandas(),
    "order_type_performance.csv": order_type_perf.toPandas(),
    "multiple_delivery_performance.csv": multi_perf.toPandas(),
    "festival_performance.csv": festival_perf.toPandas(),
    "peak_hours.csv": peak_hours.toPandas(),
    "rdd_city_summary.csv": rdd_city_df,
    "foodpulse_cleaned_delivery_data.csv": df.toPandas()
}

for filename, table in exports.items():
    table.to_csv(OUT / filename, index=False)

print("Exported files:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)


# 10. Final academic checklist

### Spark concepts
- SparkSession
- Spark DataFrame
- Schema inspection
- Cleaning and casting
- GroupBy and aggregation
- Feature engineering
- RDD map/reduceByKey
- Spark SQL
- Data export

### Dataset integrity
- Public source identified
- 45,584 × 20 raw shape verified
- No synthetic orders inserted
- No invented revenue/cancellation fields
- Distance explicitly identified as Haversine straight-line distance
- Performance bands explicitly identified as project-defined

### Source
Varunnagar, **Zomato Food Delivery Analysis**, public GitHub repository:  
https://github.com/Varunnagar/Zomato-Food-Delivery-Analysis

### Next step
Use the exported CSVs to build the **FOODPULSE multi-page Streamlit website**:
Home → Overview → Delivery Intelligence → Cities → Weather & Traffic → Vehicles → Partner Performance → Order Patterns → Spark Lab → Insights.
